# Modeling airport encodings

Predict arrival delay in minutes from origin and destination. The experiment compares
five airport encodings with Ridge, Gradient Boosting, and a standalone Decision Tree.
Each model family uses one fixed configuration across all five encodings.

## Shared setup

Draw the same 100,000-flight sample as the data-preparation notebook. Use five shared
validation folds on the 80,000 development flights and a separate 20,000-flight test set.
Each fold fits its encoders on training rows. Smoothed target encoding also cross-fits
its training values internally.

RMSE is the primary metric. MAE and R? provide additional context. Fit times are
wall-clock seconds with one thread, measured after encoding and before prediction.
Ridge fit time includes scaling. To inspect a single combination, run this setup and
its four code cells in order.

In [1]:
import sys
from pathlib import Path

project_root = next(
    path
    for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (path / "src" / "airport_encoding.py").is_file()
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import platform
from time import perf_counter

import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from scipy.sparse import csr_matrix
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeRegressor
from threadpoolctl import threadpool_limits

from src.airport_encoding import (
    AIRPORT_COLUMNS,
    RANDOM_STATE,
    TARGET_COLUMN,
    make_encoded_datasets,
    sample_flights,
)

flights = pd.read_csv(
    project_root / "data" / "flights.csv",
    usecols=[*AIRPORT_COLUMNS, TARGET_COLUMN],
)
coordinates = pd.read_csv(
    project_root / "data" / "airport_coordinates.csv", index_col="airport"
)
development, holdout = sample_flights(flights)
splitter = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
folds = list(splitter.split(development))

print(f"Development: {len(development):,} flights | Test: {len(holdout):,} flights")
print(f"Validation: {len(folds)} folds | Seed: {RANDOM_STATE}")
print(f"{platform.system()} {platform.machine()} | Python {platform.python_version()} "
      f"| scikit-learn {sklearn.__version__} | 1 thread per fit")

Development: 80,000 flights | Test: 20,000 flights
Validation: 5 folds | Seed: 42
Windows AMD64 | Python 3.13.5 | scikit-learn 1.6.1 | 1 thread per fit


In [2]:
def ridge():
    return make_pipeline(
        StandardScaler(with_mean=False),
        Ridge(alpha=1.0, solver="lsqr", fit_intercept=True, tol=1e-4, max_iter=10_000),
    )


def gradient_boosting():
    return GradientBoostingRegressor(
        loss="squared_error", n_estimators=200, learning_rate=0.05,
        max_depth=3, min_samples_leaf=5, max_features=None, subsample=1.0,
        n_iter_no_change=None, random_state=RANDOM_STATE,
    )


def decision_tree():
    return DecisionTreeRegressor(
        criterion="squared_error", splitter="best", max_depth=8,
        min_samples_leaf=50, max_features=None, random_state=RANDOM_STATE,
    )


def feature_matrix(frame, columns):
    features = frame[columns]
    if isinstance(features.dtypes.iloc[0], pd.SparseDtype):
        return features.sparse.to_coo().tocsr().astype(np.float64)
    return csr_matrix(features.to_numpy(dtype=np.float64))


def prepare_split(train, evaluation):
    datasets, _, columns = make_encoded_datasets(
        train, evaluation, coordinates, target_column=TARGET_COLUMN
    )
    y_train = train[TARGET_COLUMN].to_numpy()
    y_evaluation = evaluation[TARGET_COLUMN].to_numpy()
    prepared = {}
    for encoding, parts in datasets.items():
        if TARGET_COLUMN in columns[encoding]:
            raise ValueError(f"Target leakage in {encoding} predictors.")
        assert parts["train"].index.equals(train.index)
        assert parts["test"].index.equals(evaluation.index)
        prepared[encoding] = (
            feature_matrix(parts["train"], columns[encoding]), y_train,
            feature_matrix(parts["test"], columns[encoding]), y_evaluation,
        )
    return prepared


prepared_folds = [
    prepare_split(development.iloc[train], development.iloc[validation])
    for train, validation in folds
]
prepared_holdout = prepare_split(development, holdout)

In [3]:
def fit_and_score(make_model, split):
    X_train, y_train, X_evaluation, y_evaluation = split
    model = make_model()

    with threadpool_limits(limits=1):
        start = perf_counter()
        model.fit(X_train, y_train)
        fit_seconds = perf_counter() - start
        predictions = model.predict(X_evaluation)

    return {
        "RMSE (min)": np.sqrt(mean_squared_error(y_evaluation, predictions)),
        "MAE (min)": mean_absolute_error(y_evaluation, predictions),
        "R?": r2_score(y_evaluation, predictions),
        "Fit (s)": fit_seconds,
    }


def cross_validate(make_model, encoding):
    scores = [fit_and_score(make_model, fold[encoding]) for fold in prepared_folds]
    return pd.DataFrame(scores, index=pd.RangeIndex(1, len(scores) + 1, name="Fold"))


validation_results = {}
test_results = {}

## One-hot encoding

Each observed origin or destination airport gets its own indicator column. An unseen airport gets zeros for that field.

### Ridge

**Settings**

Alpha 1.0, LSQR, intercept, tolerance 0.0001, and up to 10,000 iterations. Scale training features without centering.

#### Training

Fit on each development fold's training flights.

In [4]:
validation_results[('one_hot', 'Ridge')] = cross_validate(ridge, 'one_hot')

#### Training performance

Report scores on each fold's validation flights.

In [5]:
display(validation_results[('one_hot', 'Ridge')].round(4))

,RMSE (min),MAE (min),R?,Fit (s)
Fold,,,,
1,44.3698,27.8137,0.0129,0.0518
2,45.2853,28.0424,0.0091,0.0384
3,45.0501,27.6488,0.0067,0.0378
4,43.8103,27.6884,0.0110,0.0347
5,44.4953,28.0054,0.0102,0.0328


#### Testing

Refit on all 80,000 development flights and predict the 20,000 test flights.

In [6]:
test_results[('one_hot', 'Ridge')] = fit_and_score(ridge, prepared_holdout['one_hot'])

#### Test performance

In [7]:
display(pd.DataFrame([test_results[('one_hot', 'Ridge')]], index=["Test"]).round(4))

,RMSE (min),MAE (min),R?,Fit (s)
Test,44.3763,27.9075,0.0135,0.0568


### Gradient Boosting

**Settings**

200 stages, learning rate 0.05, depth 3, and at least 5 flights per leaf. Squared-error loss, all rows and features available, early stopping disabled, seed 42.

#### Training

Fit on each development fold's training flights.

In [8]:
validation_results[('one_hot', 'Gradient Boosting')] = cross_validate(gradient_boosting, 'one_hot')

#### Training performance

Report scores on each fold's validation flights.

In [9]:
display(validation_results[('one_hot', 'Gradient Boosting')].round(4))

,RMSE (min),MAE (min),R?,Fit (s)
Fold,,,,
1,44.4043,27.7796,0.0114,7.1757
2,45.3019,27.9879,0.0084,5.3548
3,45.0319,27.5817,0.0075,5.1663
4,43.8247,27.6531,0.0104,5.0182
5,44.5098,27.9385,0.0095,5.7552


#### Testing

Refit on all 80,000 development flights and predict the 20,000 test flights.

In [10]:
test_results[('one_hot', 'Gradient Boosting')] = fit_and_score(gradient_boosting, prepared_holdout['one_hot'])

#### Test performance

In [11]:
display(pd.DataFrame([test_results[('one_hot', 'Gradient Boosting')]], index=["Test"]).round(4))

,RMSE (min),MAE (min),R?,Fit (s)
Test,44.4157,27.888,0.0118,11.0023


### Decision Tree

**Settings**

One tree with depth limited to 8 and at least 50 flights per leaf. Best split by squared-error reduction, all features available, seed 42.

#### Training

Fit on each development fold's training flights.

In [12]:
validation_results[('one_hot', 'Decision Tree')] = cross_validate(decision_tree, 'one_hot')

#### Training performance

Report scores on each fold's validation flights.

In [13]:
display(validation_results[('one_hot', 'Decision Tree')].round(4))

,RMSE (min),MAE (min),R?,Fit (s)
Fold,,,,
1,44.5052,27.8534,0.0069,0.1414
2,45.3955,28.0652,0.0043,0.0809
3,45.1444,27.6402,0.0026,0.0906
4,43.9327,27.7380,0.0055,0.1290
5,44.5723,27.9626,0.0067,0.0952


#### Testing

Refit on all 80,000 development flights and predict the 20,000 test flights.

In [14]:
test_results[('one_hot', 'Decision Tree')] = fit_and_score(decision_tree, prepared_holdout['one_hot'])

#### Test performance

In [15]:
display(pd.DataFrame([test_results[('one_hot', 'Decision Tree')]], index=["Test"]).round(4))

,RMSE (min),MAE (min),R?,Fit (s)
Test,44.4991,27.9366,0.0081,0.1265


## Ordinal encoding

Each airport gets an integer code within its origin or destination field. The order is alphabetical. An unseen airport gets ?1.

### Ridge

**Settings**

Alpha 1.0, LSQR, intercept, tolerance 0.0001, and up to 10,000 iterations. Scale training features without centering.

#### Training

Fit on each development fold's training flights.

In [16]:
validation_results[('ordinal', 'Ridge')] = cross_validate(ridge, 'ordinal')

#### Training performance

Report scores on each fold's validation flights.

In [17]:
display(validation_results[('ordinal', 'Ridge')].round(4))

,RMSE (min),MAE (min),R?,Fit (s)
Fold,,,,
1,44.6172,27.8872,0.0019,0.0354
2,45.4623,28.0886,0.0013,0.0326
3,45.2066,27.7118,-0.0002,0.0307
4,44.0362,27.7571,0.0008,0.0403
5,44.6938,28.0630,0.0013,0.0301


#### Testing

Refit on all 80,000 development flights and predict the 20,000 test flights.

In [18]:
test_results[('ordinal', 'Ridge')] = fit_and_score(ridge, prepared_holdout['ordinal'])

#### Test performance

In [19]:
display(pd.DataFrame([test_results[('ordinal', 'Ridge')]], index=["Test"]).round(4))

,RMSE (min),MAE (min),R?,Fit (s)
Test,44.6343,28.02,0.002,0.0444


### Gradient Boosting

**Settings**

200 stages, learning rate 0.05, depth 3, and at least 5 flights per leaf. Squared-error loss, all rows and features available, early stopping disabled, seed 42.

#### Training

Fit on each development fold's training flights.

In [20]:
validation_results[('ordinal', 'Gradient Boosting')] = cross_validate(gradient_boosting, 'ordinal')

#### Training performance

Report scores on each fold's validation flights.

In [21]:
display(validation_results[('ordinal', 'Gradient Boosting')].round(4))

,RMSE (min),MAE (min),R?,Fit (s)
Fold,,,,
1,44.3845,27.7972,0.0123,8.9797
2,45.2790,28.0043,0.0094,6.8478
3,44.9951,27.5965,0.0092,6.8540
4,43.8253,27.6577,0.0103,7.2032
5,44.4996,27.9543,0.0100,7.0840


#### Testing

Refit on all 80,000 development flights and predict the 20,000 test flights.

In [22]:
test_results[('ordinal', 'Gradient Boosting')] = fit_and_score(gradient_boosting, prepared_holdout['ordinal'])

#### Test performance

In [23]:
display(pd.DataFrame([test_results[('ordinal', 'Gradient Boosting')]], index=["Test"]).round(4))

,RMSE (min),MAE (min),R?,Fit (s)
Test,44.3884,27.884,0.013,8.6019


### Decision Tree

**Settings**

One tree with depth limited to 8 and at least 50 flights per leaf. Best split by squared-error reduction, all features available, seed 42.

#### Training

Fit on each development fold's training flights.

In [24]:
validation_results[('ordinal', 'Decision Tree')] = cross_validate(decision_tree, 'ordinal')

#### Training performance

Report scores on each fold's validation flights.

In [25]:
display(validation_results[('ordinal', 'Decision Tree')].round(4))

,RMSE (min),MAE (min),R?,Fit (s)
Fold,,,,
1,44.4255,27.8230,0.0104,0.1768
2,45.3162,28.0538,0.0077,0.1329
3,45.0478,27.6432,0.0068,0.3016
4,43.8236,27.7033,0.0104,0.1953
5,44.5373,28.0024,0.0083,0.1422


#### Testing

Refit on all 80,000 development flights and predict the 20,000 test flights.

In [26]:
test_results[('ordinal', 'Decision Tree')] = fit_and_score(decision_tree, prepared_holdout['ordinal'])

#### Test performance

In [27]:
display(pd.DataFrame([test_results[('ordinal', 'Decision Tree')]], index=["Test"]).round(4))

,RMSE (min),MAE (min),R?,Fit (s)
Test,44.4189,27.9066,0.0116,0.2426


## Frequency encoding

Each airport is represented by its share of training flights as an origin or destination. An unseen airport gets zero.

### Ridge

**Settings**

Alpha 1.0, LSQR, intercept, tolerance 0.0001, and up to 10,000 iterations. Scale training features without centering.

#### Training

Fit on each development fold's training flights.

In [28]:
validation_results[('frequency', 'Ridge')] = cross_validate(ridge, 'frequency')

#### Training performance

Report scores on each fold's validation flights.

In [29]:
display(validation_results[('frequency', 'Ridge')].round(4))

,RMSE (min),MAE (min),R?,Fit (s)
Fold,,,,
1,44.5958,27.8766,0.0028,0.0330
2,45.4358,28.0732,0.0025,0.0557
3,45.2050,27.6956,-0.0001,0.0276
4,44.0290,27.7479,0.0011,0.0236
5,44.6780,28.0419,0.0020,0.0299


#### Testing

Refit on all 80,000 development flights and predict the 20,000 test flights.

In [30]:
test_results[('frequency', 'Ridge')] = fit_and_score(ridge, prepared_holdout['frequency'])

#### Test performance

In [31]:
display(pd.DataFrame([test_results[('frequency', 'Ridge')]], index=["Test"]).round(4))

,RMSE (min),MAE (min),R?,Fit (s)
Test,44.6179,28.0,0.0028,0.0371


### Gradient Boosting

**Settings**

200 stages, learning rate 0.05, depth 3, and at least 5 flights per leaf. Squared-error loss, all rows and features available, early stopping disabled, seed 42.

#### Training

Fit on each development fold's training flights.

In [32]:
validation_results[('frequency', 'Gradient Boosting')] = cross_validate(gradient_boosting, 'frequency')

KeyboardInterrupt: 

#### Training performance

Report scores on each fold's validation flights.

In [ ]:
display(validation_results[('frequency', 'Gradient Boosting')].round(4))

,RMSE (min),MAE (min),R?,Fit (s)
Fold,,,,
1,44.3570,27.7462,0.0135,8.1498
2,45.2661,27.9861,0.0099,7.8634
3,45.0343,27.5685,0.0074,8.3734
4,43.8184,27.6482,0.0107,8.5993
5,44.5037,27.9462,0.0098,8.6743


#### Testing

Refit on all 80,000 development flights and predict the 20,000 test flights.

In [ ]:
test_results[('frequency', 'Gradient Boosting')] = fit_and_score(gradient_boosting, prepared_holdout['frequency'])

#### Test performance

In [ ]:
display(pd.DataFrame([test_results[('frequency', 'Gradient Boosting')]], index=["Test"]).round(4))

,RMSE (min),MAE (min),R?,Fit (s)
Test,44.3892,27.8469,0.013,11.3444


### Decision Tree

**Settings**

One tree with depth limited to 8 and at least 50 flights per leaf. Best split by squared-error reduction, all features available, seed 42.

#### Training

Fit on each development fold's training flights.

In [ ]:
validation_results[('frequency', 'Decision Tree')] = cross_validate(decision_tree, 'frequency')

#### Training performance

Report scores on each fold's validation flights.

In [ ]:
display(validation_results[('frequency', 'Decision Tree')].round(4))

,RMSE (min),MAE (min),R?,Fit (s)
Fold,,,,
1,44.3091,27.7425,0.0156,0.1715
2,45.3057,28.0462,0.0082,0.1864
3,45.0388,27.6080,0.0072,0.1544
4,43.7924,27.6730,0.0118,0.1532
5,44.5119,27.9587,0.0094,0.1580


#### Testing

Refit on all 80,000 development flights and predict the 20,000 test flights.

In [ ]:
test_results[('frequency', 'Decision Tree')] = fit_and_score(decision_tree, prepared_holdout['frequency'])

#### Test performance

In [ ]:
display(pd.DataFrame([test_results[('frequency', 'Decision Tree')]], index=["Test"]).round(4))

,RMSE (min),MAE (min),R?,Fit (s)
Test,44.3864,27.8787,0.0131,0.1781


## Smoothed target encoding

Each airport gets a mean arrival delay shrunk toward the training mean with smoothing weight 20. Training rows use cross-fitted values. An unseen airport gets the training mean.

### Ridge

**Settings**

Alpha 1.0, LSQR, intercept, tolerance 0.0001, and up to 10,000 iterations. Scale training features without centering.

#### Training

Fit on each development fold's training flights.

In [ ]:
validation_results[('smoothed_target', 'Ridge')] = cross_validate(ridge, 'smoothed_target')

#### Training performance

Report scores on each fold's validation flights.

In [ ]:
display(validation_results[('smoothed_target', 'Ridge')].round(4))

,RMSE (min),MAE (min),R?,Fit (s)
Fold,,,,
1,44.3947,27.8157,0.0118,0.0161
2,45.2984,28.0374,0.0085,0.0180
3,45.0416,27.6179,0.0071,0.0236
4,43.8056,27.6608,0.0112,0.0355
5,44.5050,27.9991,0.0097,0.0104


#### Testing

Refit on all 80,000 development flights and predict the 20,000 test flights.

In [ ]:
test_results[('smoothed_target', 'Ridge')] = fit_and_score(ridge, prepared_holdout['smoothed_target'])

#### Test performance

In [ ]:
display(pd.DataFrame([test_results[('smoothed_target', 'Ridge')]], index=["Test"]).round(4))

,RMSE (min),MAE (min),R?,Fit (s)
Test,44.3871,27.896,0.0131,0.0297


### Gradient Boosting

**Settings**

200 stages, learning rate 0.05, depth 3, and at least 5 flights per leaf. Squared-error loss, all rows and features available, early stopping disabled, seed 42.

#### Training

Fit on each development fold's training flights.

In [ ]:
validation_results[('smoothed_target', 'Gradient Boosting')] = cross_validate(gradient_boosting, 'smoothed_target')

#### Training performance

Report scores on each fold's validation flights.

In [ ]:
display(validation_results[('smoothed_target', 'Gradient Boosting')].round(4))

,RMSE (min),MAE (min),R?,Fit (s)
Fold,,,,
1,44.3928,27.9739,0.0119,9.1237
2,45.3101,28.0913,0.0080,11.5330
3,45.0464,27.6515,0.0069,11.8441
4,43.8082,27.7707,0.0111,9.3464
5,44.5144,28.0801,0.0093,8.9217


#### Testing

Refit on all 80,000 development flights and predict the 20,000 test flights.

In [ ]:
test_results[('smoothed_target', 'Gradient Boosting')] = fit_and_score(gradient_boosting, prepared_holdout['smoothed_target'])

#### Test performance

In [ ]:
display(pd.DataFrame([test_results[('smoothed_target', 'Gradient Boosting')]], index=["Test"]).round(4))

,RMSE (min),MAE (min),R?,Fit (s)
Test,44.4179,27.7952,0.0117,8.6847


### Decision Tree

**Settings**

One tree with depth limited to 8 and at least 50 flights per leaf. Best split by squared-error reduction, all features available, seed 42.

#### Training

Fit on each development fold's training flights.

In [ ]:
validation_results[('smoothed_target', 'Decision Tree')] = cross_validate(decision_tree, 'smoothed_target')

#### Training performance

Report scores on each fold's validation flights.

In [ ]:
display(validation_results[('smoothed_target', 'Decision Tree')].round(4))

,RMSE (min),MAE (min),R?,Fit (s)
Fold,,,,
1,44.5229,27.9934,0.0061,0.1474
2,45.3318,28.2656,0.0071,0.1209
3,45.2490,28.1480,-0.0021,0.1565
4,43.9441,27.7918,0.0050,0.1313
5,44.5743,28.1907,0.0067,0.1351


#### Testing

Refit on all 80,000 development flights and predict the 20,000 test flights.

In [ ]:
test_results[('smoothed_target', 'Decision Tree')] = fit_and_score(decision_tree, prepared_holdout['smoothed_target'])

#### Test performance

In [ ]:
display(pd.DataFrame([test_results[('smoothed_target', 'Decision Tree')]], index=["Test"]).round(4))

,RMSE (min),MAE (min),R?,Fit (s)
Test,44.4599,27.6387,0.0098,0.1521


## Geographic encoding

Origin and destination are represented by latitude and longitude from the fixed airport lookup, for four features per flight.

### Ridge

**Settings**

Alpha 1.0, LSQR, intercept, tolerance 0.0001, and up to 10,000 iterations. Scale training features without centering.

#### Training

Fit on each development fold's training flights.

In [ ]:
validation_results[('geographic', 'Ridge')] = cross_validate(ridge, 'geographic')

#### Training performance

Report scores on each fold's validation flights.

In [ ]:
display(validation_results[('geographic', 'Ridge')].round(4))

,RMSE (min),MAE (min),R?,Fit (s)
Fold,,,,
1,44.5529,27.9373,0.0047,0.0177
2,45.4052,28.1287,0.0038,0.0117
3,45.1377,27.7442,0.0029,0.0158
4,43.9710,27.7836,0.0038,0.0125
5,44.6073,28.0645,0.0052,0.0117


#### Testing

Refit on all 80,000 development flights and predict the 20,000 test flights.

In [ ]:
test_results[('geographic', 'Ridge')] = fit_and_score(ridge, prepared_holdout['geographic'])

#### Test performance

In [ ]:
display(pd.DataFrame([test_results[('geographic', 'Ridge')]], index=["Test"]).round(4))

,RMSE (min),MAE (min),R?,Fit (s)
Test,44.556,28.056,0.0055,0.0151


### Gradient Boosting

**Settings**

200 stages, learning rate 0.05, depth 3, and at least 5 flights per leaf. Squared-error loss, all rows and features available, early stopping disabled, seed 42.

#### Training

Fit on each development fold's training flights.

In [ ]:
validation_results[('geographic', 'Gradient Boosting')] = cross_validate(gradient_boosting, 'geographic')

#### Training performance

Report scores on each fold's validation flights.

In [ ]:
display(validation_results[('geographic', 'Gradient Boosting')].round(4))

,RMSE (min),MAE (min),R?,Fit (s)
Fold,,,,
1,44.3642,27.7990,0.0132,6.8735
2,45.2520,28.0039,0.0105,6.8568
3,44.9926,27.6092,0.0093,7.2090
4,43.7857,27.6648,0.0121,7.4634
5,44.4795,27.9629,0.0109,7.4498


#### Testing

Refit on all 80,000 development flights and predict the 20,000 test flights.

In [ ]:
test_results[('geographic', 'Gradient Boosting')] = fit_and_score(gradient_boosting, prepared_holdout['geographic'])

#### Test performance

In [ ]:
display(pd.DataFrame([test_results[('geographic', 'Gradient Boosting')]], index=["Test"]).round(4))

,RMSE (min),MAE (min),R?,Fit (s)
Test,44.3595,27.8792,0.0143,10.2648


### Decision Tree

**Settings**

One tree with depth limited to 8 and at least 50 flights per leaf. Best split by squared-error reduction, all features available, seed 42.

#### Training

Fit on each development fold's training flights.

In [ ]:
validation_results[('geographic', 'Decision Tree')] = cross_validate(decision_tree, 'geographic')

#### Training performance

Report scores on each fold's validation flights.

In [ ]:
display(validation_results[('geographic', 'Decision Tree')].round(4))

,RMSE (min),MAE (min),R?,Fit (s)
Fold,,,,
1,44.3635,27.7762,0.0132,0.1392
2,45.2987,28.0134,0.0085,0.1662
3,45.0147,27.6323,0.0083,0.1678
4,43.8197,27.7150,0.0106,0.1658
5,44.4879,27.9585,0.0105,0.1692


#### Testing

Refit on all 80,000 development flights and predict the 20,000 test flights.

In [ ]:
test_results[('geographic', 'Decision Tree')] = fit_and_score(decision_tree, prepared_holdout['geographic'])

#### Test performance

In [ ]:
display(pd.DataFrame([test_results[('geographic', 'Decision Tree')]], index=["Test"]).round(4))

,RMSE (min),MAE (min),R?,Fit (s)
Test,44.3573,27.8808,0.0144,0.1981


## Comparison across encodings

Compare RMSE within each model family. The validation table shows the mean and
standard deviation across folds, along with the median and range of fit times.
Test scores use the same held-out flights for every combination.

In [ ]:
validation_summary = pd.DataFrame.from_dict(
    {
        key: {
            "Mean RMSE (min)": scores["RMSE (min)"].mean(),
            "SD RMSE (min)": scores["RMSE (min)"].std(),
            "Mean MAE (min)": scores["MAE (min)"].mean(),
            "Mean R?": scores["R?"].mean(),
            "Median fit (s)": scores["Fit (s)"].median(),
            "Min fit (s)": scores["Fit (s)"].min(),
            "Max fit (s)": scores["Fit (s)"].max(),
        }
        for key, scores in validation_results.items()
    },
    orient="index",
)
test_summary = pd.DataFrame.from_dict(test_results, orient="index")
for title, summary in [("Cross-validation", validation_summary), ("Holdout test", test_summary)]:
    summary.index = pd.MultiIndex.from_tuples(summary.index, names=["Encoding", "Model"])
    print(title)
    display(summary.swaplevel().sort_index(level="Model", sort_remaining=False).round(4))

Cross-validation


Mean RMSE (min)  SD RMSE (min)  \
Model             Encoding                                          
Decision Tree     one_hot                  44.7100         0.5752   
                  ordinal                  44.6301         0.5801   
                  frequency                44.5916         0.5989   
                  smoothed_target          44.7244         0.5736   
                  geographic               44.5969         0.5783   
Gradient Boosting one_hot                  44.6145         0.5757   
                  ordinal                  44.5967         0.5643   
                  frequency                44.5959         0.5726   
                  smoothed_target          44.6144         0.5873   
                  geographic               44.5748         0.5721   
Ridge             one_hot                  44.6022         0.5830   
                  ordinal                  44.8032         0.5550   
                  frequency                44.7887         0.5520   
                  smoothed_target          44.6091         0.5841   
                  geographic               44.7348         0.5578   

                                   Mean MAE (min)  Mean R?  Median fit (s)  \
Model             Encoding                                                   
Decision Tree     one_hot                 27.8519   0.0052          0.0238   
                  ordinal                 27.8451   0.0087          0.1146   
                  frequency               27.8057   0.0105          0.1580   
                  smoothed_target         28.0779   0.0045          0.1351   
                  geographic              27.8191   0.0102          0.1662   
Gradient Boosting one_hot                 27.7882   0.0094          2.4109   
                  ordinal                 27.8020   0.0102          6.2980   
                  frequency               27.7790   0.0103          8.3734   
                  smoothed_target         27.9135   0.0094          9.3464   
                  geographic              27.8079   0.0112          7.2090   
Ridge             one_hot                 27.8397   0.0100          0.0099   
                  ordinal                 27.9015   0.0010          0.0072   
                  frequency               27.8871   0.0017          0.0171   
                  smoothed_target         27.8262   0.0097          0.0180   
                  geographic              27.9317   0.0041          0.0125   

                                   Min fit (s)  Max fit (s)  
Model             Encoding                                   
Decision Tree     one_hot               0.0237       0.0255  
                  ordinal               0.1067       0.1349  
                  frequency             0.1532       0.1864  
                  smoothed_target       0.1209       0.1565  
                  geographic            0.1392       0.1692  
Gradient Boosting one_hot               2.2111       2.9769  
                  ordinal               3.9656       6.5184  
                  frequency             7.8634       8.6743  
                  smoothed_target       8.9217      11.8441  
                  geographic            6.8568       7.4634  
Ridge             one_hot               0.0085       0.0168  
                  ordinal               0.0070       0.0079  
                  frequency             0.0105       0.0201  
                  smoothed_target       0.0104       0.0355  
                  geographic            0.0117       0.0177

Holdout test


RMSE (min)  MAE (min)      R?  Fit (s)
Model             Encoding                                               
Decision Tree     one_hot             44.4991    27.9366  0.0081   0.0350
                  ordinal             44.4189    27.9066  0.0116   0.1576
                  frequency           44.3864    27.8787  0.0131   0.1781
                  smoothed_target     44.4599    27.6387  0.0098   0.1521
                  geographic          44.3573    27.8808  0.0144   0.1981
Gradient Boosting one_hot             44.4157    27.8880  0.0118   3.2399
                  ordinal             44.3884    27.8840  0.0130   8.8972
                  frequency           44.3892    27.8469  0.0130  11.3444
                  smoothed_target     44.4179    27.7952  0.0117   8.6847
                  geographic          44.3595    27.8792  0.0143  10.2648
Ridge             one_hot             44.3763    27.9075  0.0135   0.0144
                  ordinal             44.6343    28.0200  0.0020   0.0111
                  frequency           44.6179    28.0000  0.0028   0.0284
                  smoothed_target     44.3871    27.8960  0.0131   0.0297
                  geographic          44.5560    28.0560  0.0055   0.0151

## Paired holdout comparisons

Compare each encoding with one-hot within the same model family. Negative RMSE differences favor the alternative encoding. The analysis used the same fixed models, 80,000 development flights, and 20,000 test flights as above.

Bootstrap the 365 test flight dates with replacement 20,000 times (seed 42). Each draw uses the same sampled dates for all models and includes all held-out flights on those dates. Recalculate RMSE in each draw and form percentile intervals for the paired differences. The 95% intervals below use a Bonferroni adjustment for 12 comparisons.

| Model | Encoding | RMSE change (min) | Adjusted 95% interval (min) |
| --- | --- | ---: | ---: |
| Ridge | Ordinal | +0.2581 | +0.1411 to +0.3738 |
| Ridge | Frequency | +0.2417 | +0.1337 to +0.3492 |
| Ridge | Smoothed target | +0.0108 | ?0.0217 to +0.0442 |
| Ridge | Geographic | +0.1798 | +0.0851 to +0.2790 |
| Gradient Boosting | Ordinal | ?0.0273 | ?0.0652 to +0.0097 |
| Gradient Boosting | Frequency | ?0.0265 | ?0.0635 to +0.0094 |
| Gradient Boosting | Smoothed target | +0.0022 | ?0.0690 to +0.0710 |
| Gradient Boosting | Geographic | ?0.0562 | ?0.1062 to ?0.0093 |
| Decision Tree | Ordinal | ?0.0802 | ?0.1648 to +0.0038 |
| Decision Tree | Frequency | ?0.1127 | ?0.1983 to ?0.0265 |
| Decision Tree | Smoothed target | ?0.0391 | ?0.1513 to +0.0687 |
| Decision Tree | Geographic | ?0.1418 | ?0.2357 to ?0.0465 |

One-hot had lower RMSE than ordinal, frequency, and geographic encoding for Ridge. Geographic encoding improved RMSE for Gradient Boosting and Decision Tree. Frequency also improved RMSE for Decision Tree. These differences were small. The combination with the lowest mean validation RMSE, geographic Gradient Boosting, had a test RMSE of 44.36 minutes. Predicting the development-set mean delay for every test flight gave 44.68 minutes. These results describe this split and the fixed model settings.